In [1]:
from collections import deque
import json

In [9]:
def meet_in_the_middle(grid, start, goal):
    # anzahl an zeilen und spalten bestimmen
    rows, cols = len(grid), len(grid[0])

    # start- und goalkoordinaten extrahieren
    (agent_x, agent_y) = start
    (goal_x, goal_y) = goal

    # falls start oder goal blockiert >> fehler 
    if grid[agent_x][agent_y] or grid[goal_x][goal_y]:
        raise ValueError("Start oder Ziel ist blockiert.")

    # init von disastanz- und parent-info für beide suchrichtungen
    distance_from_start = [[float('inf')] * cols for _ in range(rows)]
    distance_from_goal = [[float('inf')] * cols for _ in range(rows)]
    parent_from_start = [[None] * cols for _ in range(rows)]
    parent_from_goal = [[None] * cols for _ in range(rows)]

    # dist zum startpkt/ goalpunkt gleich 0 setzen
    distance_from_start[agent_x][agent_y] = 0
    distance_from_goal[goal_x][goal_y] = 0

    # bfs-queues für start- und ziel-richtungen
    queue_start = deque([(agent_x, agent_y)])
    queue_goal = deque([(goal_x, goal_y)])

    # bfs solange solange queue nicht leer ist
    while queue_start and queue_goal:
        # start expanded sich
        if queue_start:
            x, y = queue_start.popleft()
            # alle möglichen nachbarn von (x,y) durchgehen, in die 4 richtungen
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                nx, ny = x+dx, y+dy
                # prüfen, ob nachbar innerhalb des grids
                if 0 <= nx < rows and 0 <= ny < cols:
                    # prüfen ob der knoten frei und unbesucht (inf)
                    if not grid[nx][ny] and distance_from_start[nx][ny] == float('inf'):
                        # distanz aktualisieren und parentknoten setzen
                        distance_from_start[nx][ny] = distance_from_start[x][y] + 1
                        parent_from_start[nx][ny] = (x, y)
                        # knoten der queue hinzufügen
                        queue_start.append((nx, ny))
                        # prüfe, ob bereits besuchter knoten von goal sind >> crash
                        if distance_from_goal[nx][ny] != float('inf'):
                            # treffen sich
                            return find_path(parent_from_start, parent_from_goal, (nx, ny), start, goal)
                            
        # goal expanded sich
        if queue_goal:
            x, y = queue_goal.popleft()
            # alle nachbarn von (x, y) durchgehen in die 4 richtungen
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                nx, ny = x+dx, y+dy
                # prüfen, ob nachbar innerhalb des grids
                if 0 <= nx < rows and 0 <= ny < cols:
                    # überprüfen, ob knoten frei und unbesucht (inf)
                    if not grid[nx][ny] and distance_from_goal[nx][ny] == float('inf'):
                        # distanz aktualisieren und parentknoten setzen
                        distance_from_goal[nx][ny] = distance_from_goal[x][y] + 1
                        parent_from_goal[nx][ny] = (x, y)
                        # knoten der queue hinzufügen
                        queue_goal.append((nx, ny))
                        # prüfe ob knoten bereits von start besucht >> crash
                        if distance_from_start[nx][ny] != float('inf'):
                            # treffen sich 
                            return find_path(parent_from_start, parent_from_goal, (nx, ny), start, goal)

    # kein pfad gefunden
    return None

In [3]:
def find_path(parent_from_start, parent_from_goal, meet_point, start, goal):
    (meeting_p_x, meeting_p_y) = meet_point

    # rekonstruiere den pfad von start zu meetingpoint
    path_from_start = []
    x, y = meeting_p_x, meeting_p_y
    while (x, y) != start:
        if parent_from_start[x][y] is None:
            raise ValueError(f"Kein Parent für Knoten {(x, y)} vom Start aus gefunden.")
        x, y = parent_from_start[x][y]
        path_from_start.append((x, y))
    path_from_start.reverse()

    # rekonstruiere pfad vom meetingpoint zum goal
    path_from_goal = []
    x, y = meeting_p_x, meeting_p_y
    while (x, y) != goal:
        if parent_from_goal[x][y] is None:
            raise ValueError(f"Kein Parent für Knoten {(x, y)} vom Goal aus gefunden.")
        x, y = parent_from_goal[x][y]
        path_from_goal.append((x, y))

    # print(meeting_p_x, meeting_p_y)

    # ganzer pfad >> start zu meetingpoint, meetingpoint zu goal
    return path_from_start + [(meeting_p_x, meeting_p_y)] + path_from_goal


In [4]:
def load_grid(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    
    grid = data['grid']
    goal_x = data['goal_x'] - 1 
    goal_y = data['goal_y'] - 1
    agent_x = data['agent_x'] - 1
    agent_y = data['agent_y'] - 1

    goal = (goal_x, goal_y)
    start = (agent_x, agent_y) 
    
    return grid, start, goal

In [7]:
file_path = 'simple_grid_128.json'

grid, start, goal = load_grid(file_path)
print(start, goal)

(92, 24) (63, 108)


In [8]:
    path = meet_in_the_middle(grid, start, goal)
    if path is not None:
        print("Pfad gefunden")
        print(path)
    else:
        print("Kein Pfad gefunden.")

Pfad gefunden
[(92, 24), (91, 24), (90, 24), (89, 24), (88, 24), (87, 24), (86, 24), (85, 24), (84, 24), (83, 24), (82, 24), (81, 24), (80, 24), (79, 24), (78, 24), (77, 24), (76, 24), (75, 24), (74, 24), (73, 24), (72, 24), (72, 25), (72, 26), (72, 27), (72, 28), (72, 29), (72, 30), (72, 31), (72, 32), (72, 33), (72, 34), (72, 35), (72, 36), (72, 37), (72, 38), (72, 39), (72, 40), (72, 41), (72, 42), (72, 43), (72, 44), (72, 45), (72, 46), (72, 47), (72, 48), (72, 49), (72, 50), (72, 51), (72, 52), (72, 53), (72, 54), (72, 55), (72, 56), (72, 57), (72, 58), (72, 59), (72, 60), (72, 61), (72, 62), (72, 63), (72, 64), (72, 65), (72, 66), (72, 67), (72, 68), (72, 69), (72, 70), (72, 71), (72, 72), (72, 73), (72, 74), (72, 75), (72, 76), (72, 77), (72, 78), (72, 79), (72, 80), (72, 81), (72, 82), (72, 83), (72, 84), (72, 85), (72, 86), (72, 87), (72, 88), (72, 89), (72, 90), (72, 91), (72, 92), (72, 93), (72, 94), (72, 95), (72, 96), (72, 97), (72, 98), (72, 99), (72, 100), (72, 101), (72